# 08 · RAG from scratch with Google embeddings
**Time:** 25 min · **Goal:** understand what File Search does for you by building the four RAG steps yourself: split, embed, retrieve, answer.

Run the cells from top to bottom. **Do not share your key and do not upload private data.** Free-tier content may be used by Google to improve its products.

Get a key: https://aistudio.google.com/apikey

**Why this matters:** you control what is indexed, can inspect every chunk, and learn why an answer was wrong. Everything uses the Gemini API plus NumPy.

In [ ]:
# Setup: install the Gemini SDK and load your free AI Studio key
!pip -q install google-genai
import os, getpass
try:
    from google.colab import userdata          # Colab: left menu > Secrets > add GOOGLE_API_KEY
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("Paste your AI Studio key: ")
from google import genai
from google.genai import types
client = genai.Client(api_key=API_KEY)
# Model names retire quickly. This tries several free-tier models and uses the first that works.
CANDIDATE_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.5-flash"]
def pick_model():
    for m in CANDIDATE_MODELS:
        try:
            client.models.generate_content(model=m, contents="ping")
            print("Using model:", m)
            return m
        except Exception as e:
            print("Skipping", m, "->", str(e)[:100])
    raise RuntimeError("No model worked. Check your key, account and the model list at https://ai.google.dev/gemini-api/docs/models")
MODEL = pick_model()
import numpy as np

## Step 1 · A small knowledge base
This is **sample text for practice, not official advice.** Replace it with a real public leaflet.

In [ ]:
document = """Yellow maize leaves can be caused by too little nitrogen, waterlogged soil, or pests. Check drainage first, then ask an extension officer about fertiliser.

Fall armyworm leaves small holes and ragged edges and leaves droppings in the whorl. Inspect plants early in the morning and report outbreaks to the extension officer.

Store maize only when the grain is dry. Damp grain grows mould. Use clean bags and keep them off the floor.

Majani ya mahindi yanaweza kuwa manjano kwa sababu ya upungufu wa naitrojeni, maji kutuama au wadudu. Angalia mifereji ya maji kwanza.

Plant beans when the rains are steady. Space the seeds evenly and weed early."""
chunks = [c.strip() for c in document.split("\n\n") if c.strip()]
print(len(chunks), "chunks")

## Step 2 · Embed every chunk
An embedding turns text into a list of numbers. Similar meanings get similar numbers.

In [ ]:
EMBED_MODEL = "gemini-embedding-001"   # check ai.google.dev/gemini-api/docs/embeddings if this name fails

def embed(texts, task):
    r = client.models.embed_content(model=EMBED_MODEL, contents=texts,
                                    config=types.EmbedContentConfig(task_type=task))
    return np.array([e.values for e in r.embeddings], dtype=float)

chunk_vectors = embed(chunks, "RETRIEVAL_DOCUMENT")
print(chunk_vectors.shape)

## Step 3 · Retrieve the closest chunks
Cosine similarity measures how close two vectors point.

In [ ]:
def retrieve(question, k=2):
    q = embed([question], "RETRIEVAL_QUERY")[0]
    sims = chunk_vectors @ q / (np.linalg.norm(chunk_vectors, axis=1) * np.linalg.norm(q))
    top = np.argsort(sims)[::-1][:k]
    return [(float(sims[i]), chunks[i]) for i in top]

for score, text in retrieve("Majani ya mahindi yanageuka manjano. Sababu ni nini?"):
    print(round(score, 3), "|", text[:90])

## Step 4 · Answer using only the retrieved text

In [ ]:
def answer(question):
    hits = retrieve(question)
    context = "\n\n".join(t for _, t in hits)
    prompt = (f"Answer the farmer's question using ONLY the context. If the context does not contain the answer, say you are not sure and suggest the extension officer. "
              f"Reply in the language of the question.\n\nContext:\n{context}\n\nQuestion: {question}")
    return client.models.generate_content(model=MODEL, contents=prompt).text

print(answer("Majani ya mahindi yanageuka manjano. Sababu ni nini?"))
print("---")
print(answer("How do I treat cassava mosaic disease?"))   # not in the document: should say it is not sure

## Compare with File Search (notebook 04)
| | This notebook | File Search |
|---|---|---|
| Setup | You write about 30 lines | A few API calls |
| Control | Full: chunking, ranking, prompts | Managed for you |
| Sources | You show the chunks | Citation metadata |
| Best for | Learning, debugging, small data | Faster delivery |

## Your turn
- Change how the document is split (per sentence, per two paragraphs). Does retrieval improve?
- Ask a question in Luganda or another language. Does the right chunk still come back? Embedding quality is often *worse* in smaller languages: measure it.
- Add a similarity threshold so the agent says "not sure" when the best score is low.